Audyt metodologii 2026-09-10: wyniki historyczne unieważnione. Definicje i ograniczenia: `../docs/methodology_audit.md`. Przeliczenia lokalne: `../data/processed/audit_v2/`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Jeżeli notebook jest w folderze notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_RAW = PROJECT_ROOT / "data" / "raw"

df_path = DATA_PROCESSED / "df_model_clean_v1.parquet"
rapping_path = DATA_PROCESSED / "rapping_features_v1.parquet"
tag_classification_path = DATA_PROCESSED / "tag_classification_v1.xlsx"

df = pd.read_parquet(df_path)
# rapping features are recomputed below from the original signal
tag_classification = pd.read_excel(tag_classification_path)

print(df.shape)
print(df.index.min(), "→", df.index.max())

print(tag_classification.shape)
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.time_analysis import (validate_time, time_shift, past_mean, lagged_corr,
    rapping_starts, rapping_features, complete_window, tail_mask as post_tail_mask, coverage)

validate_time(df)
rapping = rapping_features(df["008B05154"])


In [ ]:
TARGET_COL = "008A01345"  # pył za ESP

U_COLS = {
    "U1": "008A02289",  # napięcie wtórne sekcja 1A
    "U2": "008A02290",  # napięcie wtórne sekcja 2A
    "U3": "008A02291",  # napięcie wtórne sekcja 3A
}

I_COLS = {
    "I1": "008A02267",  # prąd wtórny sekcja 1A
    "I2": "008A02268",  # prąd wtórny sekcja 2A
    "I3": "008A02269",  # prąd wtórny sekcja 3A
}

P_COLS = {
    "P1": "008A02273",  # moc sekcja 1A
    "P2": "008A02274",  # moc sekcja 2A
    "P3": "008A02275",  # moc sekcja 3A
}

RAPPING_TIME_COL = "minutes_since_rapping_3_collecting_start_0_5"
RAPPING_FLAG_COL = "is_within_5min_after_rapping_3_collecting"

required_cols = [TARGET_COL] + list(U_COLS.values()) + list(I_COLS.values()) + list(P_COLS.values())

missing = [col for col in required_cols if col not in df.columns]
missing

In [ ]:
df_eco = df.copy()

rapping_cols = [RAPPING_TIME_COL, RAPPING_FLAG_COL]

df_eco = df_eco.join(
    rapping[rapping_cols],
    how="left"
)

df_eco[rapping_cols].head()

In [ ]:
df_eco["P_total"] = df_eco[list(P_COLS.values())].sum(axis=1, min_count=3)
df_eco["U_mean"] = df_eco[list(U_COLS.values())].mean(axis=1)
df_eco["U_sum"] = df_eco[list(U_COLS.values())].sum(axis=1, min_count=3)

for sec in ["1", "2", "3"]:
    u_col = U_COLS[f"U{sec}"]
    i_col = I_COLS[f"I{sec}"]
    p_col = P_COLS[f"P{sec}"]

    df_eco[f"P{sec}_calc"] = df_eco[u_col] * df_eco[i_col] / 1000
    df_eco[f"P{sec}_error"] = df_eco[f"P{sec}_calc"] - df_eco[p_col]

df_eco["P_total_calc"] = df_eco[["P1_calc", "P2_calc", "P3_calc"]].sum(axis=1, min_count=3)
df_eco["P_total_error"] = df_eco["P_total_calc"] - df_eco["P_total"]

df_eco[
    [TARGET_COL, "P_total", "P_total_calc", "P_total_error", "U_mean"]
].describe()

In [ ]:
power_validation = []

for sec in ["1", "2", "3"]:
    p_col = P_COLS[f"P{sec}"]
    p_calc_col = f"P{sec}_calc"

    power_validation.append({
        "section": sec,
        "measured_power_col": p_col,
        "corr": df_eco[p_col].corr(df_eco[p_calc_col]),
        "mae_kw": (df_eco[p_col] - df_eco[p_calc_col]).abs().mean(),
        "bias_kw": (df_eco[p_calc_col] - df_eco[p_col]).mean(),
        "measured_mean_kw": df_eco[p_col].mean(),
        "calc_mean_kw": df_eco[p_calc_col].mean(),
    })

power_validation = pd.DataFrame(power_validation)
power_validation

In [ ]:
samples_per_min = 6

df_eco["dust_delta_1min"] = df_eco[TARGET_COL] - time_shift(df_eco[TARGET_COL], 1 * samples_per_min)
df_eco["dust_delta_3min"] = df_eco[TARGET_COL] - time_shift(df_eco[TARGET_COL], 3 * samples_per_min)

df_eco["P_delta_1min"] = df_eco["P_total"] - time_shift(df_eco["P_total"], 1 * samples_per_min)
df_eco["P_delta_3min"] = df_eco["P_total"] - time_shift(df_eco["P_total"], 3 * samples_per_min)

df_eco["U_delta_1min"] = df_eco["U_mean"] - time_shift(df_eco["U_mean"], 1 * samples_per_min)
df_eco["U_delta_3min"] = df_eco["U_mean"] - time_shift(df_eco["U_mean"], 3 * samples_per_min)

df_eco[
    ["dust_delta_3min", "P_delta_3min", "U_delta_3min"]
].describe()

In [ ]:
# lagged_corr imported from src.time_analysis; positive lag = x leads y.
max_lag_min = 10
max_lag_samples = max_lag_min * samples_per_min

corr_dust_P_delta = lagged_corr(
    df_eco["dust_delta_3min"],
    df_eco["P_delta_3min"],
    max_lag_samples=max_lag_samples
)

corr_dust_U_delta = lagged_corr(
    df_eco["dust_delta_3min"],
    df_eco["U_delta_3min"],
    max_lag_samples=max_lag_samples
)

best_P = corr_dust_P_delta.loc[corr_dust_P_delta["corr"].idxmax()]
best_U = corr_dust_U_delta.loc[corr_dust_U_delta["corr"].idxmax()]

best_P, best_U

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    corr_dust_P_delta["lag_minutes"],
    corr_dust_P_delta["corr"],
    label="corr(Δdust_3min(t), ΔP_total_3min(t+lag))"
)

plt.plot(
    corr_dust_U_delta["lag_minutes"],
    corr_dust_U_delta["corr"],
    label="corr(Δdust_3min(t), ΔU_mean_3min(t+lag))"
)

plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("Lag [min]")
plt.ylabel("Correlation")
plt.title("Lagged correlation: dust trend vs later electrical response")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
delta_windows_min = [1, 2, 3, 5]
lag_results = []

plt.figure(figsize=(11, 6))

for window_min in delta_windows_min:
    shift_n = window_min * samples_per_min
    
    dust_delta = df_eco[TARGET_COL] - time_shift(df_eco[TARGET_COL], shift_n)
    p_delta = df_eco["P_total"] - time_shift(df_eco["P_total"], shift_n)
    
    corr_df = lagged_corr(
        dust_delta,
        p_delta,
        max_lag_samples=10 * samples_per_min
    )
    
    best = corr_df.loc[corr_df["corr"].idxmax()]
    
    lag_results.append({
        "delta_window_min": window_min,
        "best_lag_min": best["lag_minutes"],
        "best_corr": best["corr"],
        "n": best["n"],
    })
    
    plt.plot(
        corr_df["lag_minutes"],
        corr_df["corr"],
        label=f"Δ window = {window_min} min"
    )

plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("Lag [min]")
plt.ylabel("Correlation")
plt.title("Lagged correlation: Δdust vs ΔP_total for different delta windows")
plt.legend()
plt.grid(True)
plt.show()

lag_results = pd.DataFrame(lag_results)
lag_results

In [ ]:
rapping.columns

In [ ]:
RAPPING_START_COL = "rapping_3_collecting_start"

# Dołączamy start strzepywania, jeśli jeszcze nie został dołączony
if RAPPING_START_COL not in df_eco.columns:
    df_eco = df_eco.join(rapping[[RAPPING_START_COL]], how="left")

rapping_starts = df_eco.index[df_eco[RAPPING_START_COL] == 1]

print("Number of rapping starts:", len(rapping_starts))
print("First events:")
print(rapping_starts[:10])

In [ ]:
pre_min = 5
post_min = 15

pre_samples = pre_min * samples_per_min
post_samples = post_min * samples_per_min

signals_for_event = [
    TARGET_COL,
    "P_total",
    "U_mean",
    "U_sum",
    *list(U_COLS.values()),
    *list(I_COLS.values()),
    *list(P_COLS.values()),
]

event_windows = []

event_rejections = {"incomplete": 0, "overlap": 0}
for event_id, event_time in enumerate(rapping_starts):
    window = complete_window(df_eco[signals_for_event], event_time, -pre_min * 60, post_min * 60)
    if window is None:
        event_rejections['incomplete'] += 1
        continue
    # Exclude overlapping same-tag responses, including preceding 15 min.
    neighbours = (rapping_starts > event_time - pd.Timedelta(minutes=15)) & (rapping_starts < event_time + pd.Timedelta(minutes=15))
    if neighbours.sum() > 1:
        event_rejections['overlap'] += 1
        continue
    window['event_id'] = event_id
    window['event_time'] = event_time
    window['t_rel_min'] = (window.index - event_time).total_seconds() / 60
    event_windows.append(window)
if not event_windows:
    raise ValueError('No complete, isolated ECO windows')
print(event_rejections)
events_df = pd.concat(event_windows, axis=0)

print(events_df.shape)
events_df.head()

In [ ]:
avg_response = (
    events_df
    .groupby("t_rel_min")[signals_for_event]
    .mean()
)

q25_response = (
    events_df
    .groupby("t_rel_min")[signals_for_event]
    .quantile(0.25)
)

q75_response = (
    events_df
    .groupby("t_rel_min")[signals_for_event]
    .quantile(0.75)
)

avg_response.head()

In [ ]:
plt.figure(figsize=(11, 5))

x = avg_response.index
y = avg_response[TARGET_COL]

plt.plot(x, y, label="Mean dust")
plt.fill_between(
    x,
    q25_response[TARGET_COL],
    q75_response[TARGET_COL],
    alpha=0.2,
    label="Q25–Q75"
)

plt.axvline(0, linestyle="--", linewidth=1, label="Rapping start")
plt.xlabel("Time relative to rapping start [min]")
plt.ylabel("Dust concentration [mg/Nm³]")
plt.title("Average dust response after rapping event")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(11, 5))

plt.plot(avg_response.index, avg_response["P_total"], label="P_total [kW]")
plt.plot(avg_response.index, avg_response["U_mean"], label="U_mean [kV]")

plt.axvline(0, linestyle="--", linewidth=1, label="Rapping start")
plt.xlabel("Time relative to rapping start [min]")
plt.ylabel("Mean value")
plt.title("Average electrical response after rapping event")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
baseline_window = (-5, -1)

event_relative = []

for event_id, group in events_df.groupby("event_id"):
    baseline_mask = (
        (group["t_rel_min"] >= baseline_window[0])
        & (group["t_rel_min"] <= baseline_window[1])
    )
    
    if baseline_mask.sum() == 0:
        continue
    
    baseline = group.loc[baseline_mask, signals_for_event].mean()
    
    rel = group.copy()
    for col in signals_for_event:
        rel[col + "_rel"] = rel[col] - baseline[col]
    
    event_relative.append(rel)

events_rel_df = pd.concat(event_relative, axis=0)

rel_cols = [col + "_rel" for col in signals_for_event]

avg_rel_response = (
    events_rel_df
    .groupby("t_rel_min")[rel_cols]
    .mean()
)

q25_rel_response = (
    events_rel_df
    .groupby("t_rel_min")[rel_cols]
    .quantile(0.25)
)

q75_rel_response = (
    events_rel_df
    .groupby("t_rel_min")[rel_cols]
    .quantile(0.75)
)

avg_rel_response.head()

In [ ]:
plt.figure(figsize=(11, 5))

x = avg_rel_response.index

plt.plot(x, avg_rel_response[TARGET_COL + "_rel"], label="Δdust vs baseline")
plt.fill_between(
    x,
    q25_rel_response[TARGET_COL + "_rel"],
    q75_rel_response[TARGET_COL + "_rel"],
    alpha=0.2
)

plt.axvline(0, linestyle="--", linewidth=1, label="Rapping start")
plt.axhline(0, linestyle=":", linewidth=1)
plt.xlabel("Time relative to rapping start [min]")
plt.ylabel("Δ dust [mg/Nm³]")
plt.title("Average relative dust response after rapping")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(11, 5))

plt.plot(x, avg_rel_response["P_total_rel"], label="ΔP_total vs baseline [kW]")
plt.plot(x, avg_rel_response["U_mean_rel"], label="ΔU_mean vs baseline [kV]")

plt.axvline(0, linestyle="--", linewidth=1, label="Rapping start")
plt.axhline(0, linestyle=":", linewidth=1)
plt.xlabel("Time relative to rapping start [min]")
plt.ylabel("Δ value")
plt.title("Average relative electrical response after rapping")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
event_metrics = []

for event_id, group in events_rel_df.groupby("event_id"):
    after_0_5 = group[(group["t_rel_min"] >= 0) & (group["t_rel_min"] <= 5)]
    after_0_15 = group[(group["t_rel_min"] >= 0) & (group["t_rel_min"] <= 15)]
    
    if len(after_0_5) == 0 or len(after_0_15) == 0:
        continue
    
    dust_rel_col = TARGET_COL + "_rel"
    P_rel_col = "P_total_rel"
    U_rel_col = "U_mean_rel"
    
    dust_peak_idx = after_0_5[dust_rel_col].idxmax()
    P_peak_idx = after_0_15[P_rel_col].idxmax()
    U_peak_idx = after_0_15[U_rel_col].idxmax()
    
    event_metrics.append({
        "event_id": event_id,
        "event_time": group["event_time"].iloc[0],
        "dust_peak_0_5min_rel": after_0_5.loc[dust_peak_idx, dust_rel_col],
        "time_to_dust_peak_min": after_0_5.loc[dust_peak_idx, "t_rel_min"],
        "P_peak_0_15min_rel": after_0_15.loc[P_peak_idx, P_rel_col],
        "time_to_P_peak_min": after_0_15.loc[P_peak_idx, "t_rel_min"],
        "U_peak_0_15min_rel": after_0_15.loc[U_peak_idx, U_rel_col],
        "time_to_U_peak_min": after_0_15.loc[U_peak_idx, "t_rel_min"],
    })

event_metrics = pd.DataFrame(event_metrics)

event_metrics.describe()

In [ ]:
summary_cols = [
    "dust_peak_0_5min_rel",
    "time_to_dust_peak_min",
    "P_peak_0_15min_rel",
    "time_to_P_peak_min",
    "U_peak_0_15min_rel",
    "time_to_U_peak_min",
]

event_metrics[summary_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])

In [ ]:
plt.figure(figsize=(7, 6))

plt.scatter(
    event_metrics["dust_peak_0_5min_rel"],
    event_metrics["P_peak_0_15min_rel"],
    alpha=0.5,
    s=20
)

plt.xlabel("Dust peak after rapping [mg/Nm³ above baseline]")
plt.ylabel("Power response after rapping [kW above baseline]")
plt.title("Rapping events: dust peak vs ECO power response")
plt.grid(True)
plt.show()

corr_peak_response = event_metrics["dust_peak_0_5min_rel"].corr(
    event_metrics["P_peak_0_15min_rel"]
)

corr_peak_response

In [ ]:
corr_peak_response = event_metrics["dust_peak_0_5min_rel"].corr(
    event_metrics["P_peak_0_15min_rel"]
)

corr_peak_response

In [ ]:
event_metrics.sort_values("P_peak_0_15min_rel", ascending=False).head(15)

In [ ]:
typical_events = event_metrics[
    (event_metrics["P_peak_0_15min_rel"] >= 0)
    & (event_metrics["P_peak_0_15min_rel"] < event_metrics["P_peak_0_15min_rel"].quantile(0.98))
].copy()

print("All events:", len(event_metrics))
print("Typical events:", len(typical_events))
print("Removed:", len(event_metrics) - len(typical_events))

typical_events[summary_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])

In [ ]:
corr_peak_response_typical = typical_events["dust_peak_0_5min_rel"].corr(
    typical_events["P_peak_0_15min_rel"]
)

corr_peak_response_typical

In [ ]:
plt.figure(figsize=(7, 6))

plt.scatter(
    typical_events["dust_peak_0_5min_rel"],
    typical_events["P_peak_0_15min_rel"],
    alpha=0.5,
    s=20
)

plt.xlabel("Dust peak after rapping [mg/Nm³ above baseline]")
plt.ylabel("Power response after rapping [kW above baseline]")
plt.title("Typical rapping events: dust peak vs ECO power response")
plt.grid(True)
plt.show()

In [ ]:
POWER_RESPONSE_THRESHOLD_KW = 5.0
DUST_RESPONSE_THRESHOLD_MG = 5.0

duration_metrics = []

for event_id, group in events_rel_df.groupby("event_id"):
    after = group[(group["t_rel_min"] >= 0) & (group["t_rel_min"] < 15)].copy()
    
    if len(after) == 0:
        continue
    
    p_rel = after["P_total_rel"]
    dust_rel = after[TARGET_COL + "_rel"]
    
    # czas próbkowania w godzinach: 10 s = 10/3600 h
    sample_hours = 10 / 3600
    
    # dodatkowa energia tylko tam, gdzie P_rel > 0
    extra_energy_kwh = p_rel.clip(lower=0).sum() * sample_hours
    
    power_above = p_rel > POWER_RESPONSE_THRESHOLD_KW
    dust_above = dust_rel > DUST_RESPONSE_THRESHOLD_MG
    
    power_duration_min = power_above.sum() / samples_per_min
    dust_duration_min = dust_above.sum() / samples_per_min
    
    duration_metrics.append({
        "event_id": event_id,
        "event_time": group["event_time"].iloc[0],
        "extra_energy_kwh_0_15min": extra_energy_kwh,
        "power_duration_above_5kw_min": power_duration_min,
        "dust_duration_above_5mg_min": dust_duration_min,
        "power_minus_dust_duration_min": power_duration_min - dust_duration_min,
    })

duration_metrics = pd.DataFrame(duration_metrics)

duration_metrics.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])

In [ ]:
event_analysis = event_metrics.merge(
    duration_metrics,
    on=["event_id", "event_time"],
    how="left"
)

event_analysis.head()

In [ ]:
event_analysis[
    [
        "dust_peak_0_5min_rel",
        "P_peak_0_15min_rel",
        "U_peak_0_15min_rel",
        "time_to_dust_peak_min",
        "time_to_P_peak_min",
        "extra_energy_kwh_0_15min",
        "power_duration_above_5kw_min",
        "dust_duration_above_5mg_min",
        "power_minus_dust_duration_min",
    ]
].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])

In [ ]:
plt.figure(figsize=(7, 6))

plt.scatter(
    event_analysis["dust_duration_above_5mg_min"],
    event_analysis["power_duration_above_5kw_min"],
    alpha=0.5,
    s=20
)

max_val = max(
    event_analysis["dust_duration_above_5mg_min"].max(),
    event_analysis["power_duration_above_5kw_min"].max()
)

plt.plot([0, max_val], [0, max_val], linestyle="--", linewidth=1)

plt.xlabel("Dust duration above +5 mg/Nm³ [min]")
plt.ylabel("Power duration above +5 kW [min]")
plt.title("Rapping events: dust duration vs ECO power response duration")
plt.grid(True)
plt.show()

In [ ]:
candidate_overreaction = event_analysis[
    (event_analysis["dust_peak_0_5min_rel"] < event_analysis["dust_peak_0_5min_rel"].quantile(0.75))
    & (event_analysis["power_duration_above_5kw_min"] > event_analysis["power_duration_above_5kw_min"].quantile(0.75))
    & (event_analysis["extra_energy_kwh_0_15min"] > event_analysis["extra_energy_kwh_0_15min"].quantile(0.75))
].copy()

candidate_overreaction = candidate_overreaction.sort_values(
    "extra_energy_kwh_0_15min",
    ascending=False
)

print("Candidate overreaction events:", len(candidate_overreaction))
candidate_overreaction.head(20)

In [ ]:
def plot_rapping_event(event_id):
    group = events_rel_df[events_rel_df["event_id"] == event_id].copy()
    
    if group.empty:
        print("No event:", event_id)
        return
    
    event_time = group["event_time"].iloc[0]
    
    fig, ax1 = plt.subplots(figsize=(12, 5))
    
    ax1.plot(
        group["t_rel_min"],
        group[TARGET_COL + "_rel"],
        label="Δdust [mg/Nm³]",
    )
    ax1.set_xlabel("Time relative to rapping start [min]")
    ax1.set_ylabel("Δdust [mg/Nm³]")
    ax1.axvline(0, linestyle="--", linewidth=1)
    ax1.axhline(0, linestyle=":", linewidth=1)
    
    ax2 = ax1.twinx()
    ax2.plot(
        group["t_rel_min"],
        group["P_total_rel"],
        label="ΔP_total [kW]",
        linestyle="--",
    )
    ax2.plot(
        group["t_rel_min"],
        group["U_mean_rel"],
        label="ΔU_mean [kV]",
        linestyle=":",
    )
    ax2.set_ylabel("ΔP [kW] / ΔU [kV]")
    
    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")
    
    plt.title(f"Rapping event {event_id} at {event_time}")
    plt.grid(True)
    plt.show()

In [ ]:
for eid in candidate_overreaction["event_id"].head(5):
    plot_rapping_event(eid)

In [ ]:
# Filtr diagnostyczny: usuwamy najbardziej ekstremalne 2% reakcji energetycznych
energy_upper = event_analysis["extra_energy_kwh_0_15min"].quantile(0.98)
power_upper = event_analysis["P_peak_0_15min_rel"].quantile(0.98)

typical_event_analysis = event_analysis[
    (event_analysis["extra_energy_kwh_0_15min"] <= energy_upper)
    & (event_analysis["P_peak_0_15min_rel"] <= power_upper)
].copy()

print("All events:", len(event_analysis))
print("Typical events:", len(typical_event_analysis))
print("Removed events:", len(event_analysis) - len(typical_event_analysis))

typical_event_analysis[
    [
        "dust_peak_0_5min_rel",
        "P_peak_0_15min_rel",
        "U_peak_0_15min_rel",
        "time_to_dust_peak_min",
        "time_to_P_peak_min",
        "extra_energy_kwh_0_15min",
        "power_duration_above_5kw_min",
        "dust_duration_above_5mg_min",
        "power_minus_dust_duration_min",
    ]
].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])

In [ ]:
plt.figure(figsize=(7, 6))

plt.scatter(
    typical_event_analysis["dust_duration_above_5mg_min"],
    typical_event_analysis["power_duration_above_5kw_min"],
    alpha=0.5,
    s=20
)

max_val = max(
    typical_event_analysis["dust_duration_above_5mg_min"].max(),
    typical_event_analysis["power_duration_above_5kw_min"].max()
)

plt.plot([0, max_val], [0, max_val], linestyle="--", linewidth=1)

plt.xlabel("Dust duration above +5 mg/Nm³ [min]")
plt.ylabel("Power duration above +5 kW [min]")
plt.title("Typical rapping events: dust duration vs ECO power response duration")
plt.grid(True)
plt.show()

In [ ]:
fraction_power_longer = (
    typical_event_analysis["power_duration_above_5kw_min"]
    > typical_event_analysis["dust_duration_above_5mg_min"]
).mean()

fraction_power_longer

In [ ]:
TAIL_POWER_THRESHOLD_KW = 5.0
TAIL_DUST_THRESHOLD_MG = 5.0

tail_energy_rows = []

for event_id, group in events_rel_df.groupby("event_id"):
    after = group[(group["t_rel_min"] >= 0) & (group["t_rel_min"] < 15)].copy()
    
    if len(after) == 0:
        continue
    
    p_rel = after["P_total_rel"]
    dust_rel = after[TARGET_COL + "_rel"]
    
    sample_hours = 10 / 3600
    
    # energia "ogonowa": moc podwyższona, ale pył już nie jest podwyższony
    coincidence_mask = (p_rel > TAIL_POWER_THRESHOLD_KW) & (dust_rel <= TAIL_DUST_THRESHOLD_MG)
    tail_mask = post_tail_mask(p_rel, dust_rel, TAIL_POWER_THRESHOLD_KW, TAIL_DUST_THRESHOLD_MG)
    
    tail_energy_kwh = p_rel.where(tail_mask, 0).clip(lower=0).sum() * sample_hours
    tail_duration_min = tail_mask.sum() / samples_per_min
    
    tail_energy_rows.append({
        "event_id": event_id,
        "tail_energy_kwh_0_15min": tail_energy_kwh,
        "tail_duration_min": tail_duration_min,
        "coincidence_energy_kwh_0_15min": p_rel.where(coincidence_mask, 0).sum() * sample_hours,
        "dust_excursion_observed": bool((dust_rel > TAIL_DUST_THRESHOLD_MG).any()),
        "recovery_observed": bool((dust_rel > TAIL_DUST_THRESHOLD_MG).any() and
                                  not (dust_rel.iloc[-6:] > TAIL_DUST_THRESHOLD_MG).any()),
    })

tail_energy = pd.DataFrame(tail_energy_rows)

event_analysis = event_analysis.merge(
    tail_energy,
    on="event_id",
    how="left"
)

typical_event_analysis = typical_event_analysis.merge(
    tail_energy,
    on="event_id",
    how="left"
)

typical_event_analysis[
    [
        "extra_energy_kwh_0_15min",
        "tail_energy_kwh_0_15min",
        "tail_duration_min",
    ]
].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])

In [ ]:
typical_event_analysis["tail_energy_share"] = (
    typical_event_analysis["tail_energy_kwh_0_15min"]
    / typical_event_analysis["extra_energy_kwh_0_15min"].replace(0, np.nan)
)

typical_event_analysis[
    ["tail_energy_share"]
].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])

In [ ]:
candidate_tail_saving = typical_event_analysis[
    (typical_event_analysis["tail_energy_share"] > typical_event_analysis["tail_energy_share"].quantile(0.75))
    & (typical_event_analysis["tail_duration_min"] > typical_event_analysis["tail_duration_min"].quantile(0.75))
    & (typical_event_analysis["dust_peak_0_5min_rel"] < typical_event_analysis["dust_peak_0_5min_rel"].quantile(0.75))
].copy()

candidate_tail_saving = candidate_tail_saving.sort_values(
    "tail_energy_kwh_0_15min",
    ascending=False
)

print("Candidate tail-saving events:", len(candidate_tail_saving))

candidate_tail_saving[
    [
        "event_id",
        "event_time",
        "dust_peak_0_5min_rel",
        "P_peak_0_15min_rel",
        "extra_energy_kwh_0_15min",
        "tail_energy_kwh_0_15min",
        "tail_energy_share",
        "tail_duration_min",
        "power_duration_above_5kw_min",
        "dust_duration_above_5mg_min",
    ]
].head(20)

In [ ]:
for eid in candidate_tail_saving["event_id"].head(5):
    plot_rapping_event(eid)

In [ ]:
key_summary = {
    "n_events_all": len(event_analysis),
    "n_events_typical": len(typical_event_analysis),
    
    "median_dust_peak_mg": typical_event_analysis["dust_peak_0_5min_rel"].median(),
    "median_power_peak_kw": typical_event_analysis["P_peak_0_15min_rel"].median(),
    "median_u_peak_kv": typical_event_analysis["U_peak_0_15min_rel"].median(),
    
    "median_time_to_dust_peak_min": typical_event_analysis["time_to_dust_peak_min"].median(),
    "median_time_to_power_peak_min": typical_event_analysis["time_to_P_peak_min"].median(),
    
    "median_extra_energy_kwh": typical_event_analysis["extra_energy_kwh_0_15min"].median(),
    "median_tail_energy_kwh": typical_event_analysis["tail_energy_kwh_0_15min"].median(),
    "median_tail_energy_share": typical_event_analysis["tail_energy_share"].median(),
    
    "median_power_duration_min": typical_event_analysis["power_duration_above_5kw_min"].median(),
    "median_dust_duration_min": typical_event_analysis["dust_duration_above_5mg_min"].median(),
    "median_power_minus_dust_duration_min": typical_event_analysis["power_minus_dust_duration_min"].median(),
    
    "fraction_power_longer_than_dust": (
        typical_event_analysis["power_duration_above_5kw_min"]
        > typical_event_analysis["dust_duration_above_5mg_min"]
    ).mean(),
    
    "n_candidate_tail_saving": len(candidate_tail_saving),
}

key_summary = pd.Series(key_summary)
key_summary

Wyniki opisują zależności obserwacyjne. Opóźnienie korelacji nie identyfikuje
przyczynowości ani trybu ECO. Energia ogonowa: dodatnia nadwyżka mocy ponad
bazę, przy ΔP > 5 kW, po ostatnim przekroczeniu Δpyłu > 5 mg/Nm³ i co najmniej
60 s obserwowanego powrotu, w [0,15 min). Brak powrotu oznacza cenzurowanie.
Próg 5 kW wybiera próbki; całkujemy ΔP, nie ΔP−5. Osobny wskaźnik coincidence
nie wymaga wcześniejszego wzrostu pyłu. Żaden wskaźnik nie jest dowodem
zbędnego zużycia ani górnym ograniczeniem oszczędności. Filtr „typical” wybiera
zdarzenia według wyniku i służy wyłącznie analizie wrażliwości.


In [ ]:
# Alias dla względnych mocy sekcji
P1_REL = P_COLS["P1"] + "_rel"
P2_REL = P_COLS["P2"] + "_rel"
P3_REL = P_COLS["P3"] + "_rel"

U1_REL = U_COLS["U1"] + "_rel"
U2_REL = U_COLS["U2"] + "_rel"
U3_REL = U_COLS["U3"] + "_rel"

print(P1_REL, P2_REL, P3_REL)
print(U1_REL, U2_REL, U3_REL)

In [ ]:
plt.figure(figsize=(11, 5))

plt.plot(avg_rel_response.index, avg_rel_response[P1_REL], label="ΔP1")
plt.plot(avg_rel_response.index, avg_rel_response[P2_REL], label="ΔP2")
plt.plot(avg_rel_response.index, avg_rel_response[P3_REL], label="ΔP3")

plt.axvline(0, linestyle="--", linewidth=1, label="Rapping start")
plt.axhline(0, linestyle=":", linewidth=1)

plt.xlabel("Time relative to rapping start [min]")
plt.ylabel("Δ section power [kW]")
plt.title("Average relative section power response after rapping")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(11, 5))

plt.plot(avg_rel_response.index, avg_rel_response[U1_REL], label="ΔU1")
plt.plot(avg_rel_response.index, avg_rel_response[U2_REL], label="ΔU2")
plt.plot(avg_rel_response.index, avg_rel_response[U3_REL], label="ΔU3")

plt.axvline(0, linestyle="--", linewidth=1, label="Rapping start")
plt.axhline(0, linestyle=":", linewidth=1)

plt.xlabel("Time relative to rapping start [min]")
plt.ylabel("Δ section voltage [kV]")
plt.title("Average relative section voltage response after rapping")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
section_tail_rows = []

for event_id, group in events_rel_df.groupby("event_id"):
    after = group[(group["t_rel_min"] >= 0) & (group["t_rel_min"] < 15)].copy()
    
    if len(after) == 0:
        continue
    
    dust_rel = after[TARGET_COL + "_rel"]
    sample_hours = 10 / 3600
    
    tail_mask = post_tail_mask(after["P_total_rel"], dust_rel, TAIL_POWER_THRESHOLD_KW, TAIL_DUST_THRESHOLD_MG)
    
    row = {
        "event_id": event_id,
        "tail_energy_P1_kwh": after[P1_REL].where(tail_mask, 0).sum() * sample_hours,
        "tail_energy_P2_kwh": after[P2_REL].where(tail_mask, 0).sum() * sample_hours,
        "tail_energy_P3_kwh": after[P3_REL].where(tail_mask, 0).sum() * sample_hours,
    }
    
    row["tail_energy_sections_sum_kwh"] = (
        row["tail_energy_P1_kwh"]
        + row["tail_energy_P2_kwh"]
        + row["tail_energy_P3_kwh"]
    )
    
    section_tail_rows.append(row)

section_tail_energy = pd.DataFrame(section_tail_rows)

typical_event_analysis = typical_event_analysis.merge(
    section_tail_energy,
    on="event_id",
    how="left"
)

typical_event_analysis[
    [
        "tail_energy_P1_kwh",
        "tail_energy_P2_kwh",
        "tail_energy_P3_kwh",
        "tail_energy_sections_sum_kwh",
        "tail_energy_kwh_0_15min",
    ]
].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])

In [ ]:
eps = 1e-9

typical_event_analysis["tail_share_P1"] = (
    typical_event_analysis["tail_energy_P1_kwh"]
    / typical_event_analysis["tail_energy_sections_sum_kwh"].replace(0, np.nan)
)

typical_event_analysis["tail_share_P2"] = (
    typical_event_analysis["tail_energy_P2_kwh"]
    / typical_event_analysis["tail_energy_sections_sum_kwh"].replace(0, np.nan)
)

typical_event_analysis["tail_share_P3"] = (
    typical_event_analysis["tail_energy_P3_kwh"]
    / typical_event_analysis["tail_energy_sections_sum_kwh"].replace(0, np.nan)
)

typical_event_analysis[
    ["tail_share_P1", "tail_share_P2", "tail_share_P3"]
].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])

In [ ]:
section_share_mean = typical_event_analysis[
    ["tail_share_P1", "tail_share_P2", "tail_share_P3"]
].mean()

section_share_median = typical_event_analysis[
    ["tail_share_P1", "tail_share_P2", "tail_share_P3"]
].median()

section_share_df = pd.DataFrame({
    "mean_share": section_share_mean,
    "median_share": section_share_median,
})

section_share_df

In [ ]:
section_share_df.plot(kind="bar", figsize=(7, 5))
plt.ylabel("Share of section tail energy")
plt.title("Section contribution to post-rapping tail energy")
plt.grid(axis="y")
plt.show()

In [ ]:
# Total observed-period tail-energy summary for all and typical rapping events

def summarize_tail_energy(events_df, label):
    total_events = len(events_df)

    total_extra_energy = events_df["extra_energy_kwh_0_15min"].sum()
    total_tail_energy = events_df["tail_energy_kwh_0_15min"].sum()

    median_extra_energy = events_df["extra_energy_kwh_0_15min"].median()
    median_tail_energy = events_df["tail_energy_kwh_0_15min"].median()

    mean_extra_energy = events_df["extra_energy_kwh_0_15min"].mean()
    mean_tail_energy = events_df["tail_energy_kwh_0_15min"].mean()

    total_tail_share = total_tail_energy / total_extra_energy if total_extra_energy > 0 else np.nan

    return {
        "dataset": label,
        "n_events": total_events,
        "total_extra_energy_kwh": total_extra_energy,
        "total_tail_energy_kwh": total_tail_energy,
        "total_tail_energy_share": total_tail_share,
        "mean_extra_energy_kwh_per_event": mean_extra_energy,
        "mean_tail_energy_kwh_per_event": mean_tail_energy,
        "median_extra_energy_kwh_per_event": median_extra_energy,
        "median_tail_energy_kwh_per_event": median_tail_energy,
    }


observed_tail_summary = pd.DataFrame([
    summarize_tail_energy(event_analysis, "all_events"),
    summarize_tail_energy(typical_event_analysis, "typical_events"),
])

observed_tail_summary

In [ ]:
energy_prices_pln_per_kwh = [0.5, 0.8, 1.0, 1.5]

cost_rows = []

for _, row in observed_tail_summary.iterrows():
    for price in energy_prices_pln_per_kwh:
        cost_rows.append({
            "dataset": row["dataset"],
            "energy_price_pln_per_kwh": price,
            "total_extra_energy_cost_pln": row["total_extra_energy_kwh"] * price,
            "total_tail_energy_cost_pln": row["total_tail_energy_kwh"] * price,
        })

observed_tail_cost = pd.DataFrame(cost_rows)
observed_tail_cost

In [ ]:
coverage_summary = coverage(df_eco[signals_for_event])
data_duration_days = coverage_summary['observed_days']
coverage_summary


In [ ]:
# Disabled: representative exposure and missing-event correction are unverified.
# Only enable for an explicitly hypothetical scenario, never a savings estimate.
ALLOW_HYPOTHETICAL_ANNUALIZATION = False
annual_factor = 365 / data_duration_days if ALLOW_HYPOTHETICAL_ANNUALIZATION and data_duration_days > 0 else np.nan

annual_tail_summary = observed_tail_summary.copy()
annual_tail_summary["data_duration_days"] = data_duration_days
annual_tail_summary["annual_factor"] = annual_factor

annual_tail_summary["annual_extra_energy_kwh_est"] = (
    annual_tail_summary["total_extra_energy_kwh"] * annual_factor
)

annual_tail_summary["annual_tail_energy_kwh_est"] = (
    annual_tail_summary["total_tail_energy_kwh"] * annual_factor
)

annual_tail_summary

In [ ]:
annual_cost_rows = []

for _, row in annual_tail_summary.iterrows():
    for price in energy_prices_pln_per_kwh:
        annual_cost_rows.append({
            "dataset": row["dataset"],
            "energy_price_pln_per_kwh": price,
            "annual_extra_energy_cost_pln_est": row["annual_extra_energy_kwh_est"] * price,
            "annual_tail_energy_cost_pln_est": row["annual_tail_energy_kwh_est"] * price,
        })

annual_tail_cost = pd.DataFrame(annual_cost_rows)
annual_tail_cost

In [ ]:
reduction_scenarios = [0.10, 0.25, 0.50, 0.75]

scenario_rows = []

for _, row in annual_tail_summary.iterrows():
    for reduction in reduction_scenarios:
        for price in energy_prices_pln_per_kwh:
            scenario_rows.append({
                "dataset": row["dataset"],
                "tail_reduction_fraction": reduction,
                "energy_price_pln_per_kwh": price,
                "annual_energy_saving_kwh_est": row["annual_tail_energy_kwh_est"] * reduction,
                "annual_cost_saving_pln_est": row["annual_tail_energy_kwh_est"] * reduction * price,
            })

saving_scenarios = pd.DataFrame(scenario_rows)
saving_scenarios

In [ ]:
typical_annual = annual_tail_summary[
    annual_tail_summary["dataset"] == "typical_events"
].iloc[0]

compact_summary = pd.DataFrame({
    "metric": [
        "Number of typical rapping events in dataset",
        "Dataset duration [days]",
        "Total additional post-rapping energy [kWh/observed period]",
        "Total tail energy [kWh/observed period]",
        "Tail energy share [-]",
        "Estimated annual tail energy [kWh/year]",
    ],
    "value": [
        typical_annual["n_events"],
        typical_annual["data_duration_days"],
        typical_annual["total_extra_energy_kwh"],
        typical_annual["total_tail_energy_kwh"],
        typical_annual["total_tail_energy_share"],
        typical_annual["annual_tail_energy_kwh_est"],
    ]
})

compact_summary